# Pilot (Design 11, 7.4, 5.4) and the Pilot reading

BanglaIndoorVQA main paper. The pilot ran the 100 PILOT items under the 9 conditions without option sets for all three models on 2026-10-14 (cells 0b to 6, DESIGN_ADDENDUM F). This notebook now runs the **Pilot reading**: the four forced-choice conditions (N-FCB, N-FCB0 on the 20 naming items with `in_pclust = true`; N-FCE, N-FCE0 on the 20 naming items in P-FCE), the scoring of every pilot answer with scorer 0.2.1-stage2, and the reading of Gates C and D (DESIGN_ADDENDUM B10, F5, H5; decisions D164 to D168).

**Order of work for the Pilot reading.** Never run two billed or GPU cells at the same time.

| Step | Runtime | Cells |
|---|---|---|
| 1. Push the two files of the Pilot reading chat | any | Cell 9 of the old notebook with `UPLOAD_FROM_COMPUTER = True` (choose `pilot.py` and `pilot.ipynb`); then open this notebook (File, Upload notebook) |
| 2. Session A: inputs | any runtime | 0 with `SESSION = "gemini"`: `ALL INPUTS PASS`. Then Cell 5 of `notebooks/asset11_options.ipynb` (after its own Cell 0): `FINAL FILES OK. P-FCE = 454 items` |
| 3. Session A: forced-choice dry run | same session | 8a (no cost). It must end `FC-VERIFY: PASS` |
| 4. Session A: Gemini forced choice | same session | 8b (80 calls, about 0.04 USD) |
| 5. Session B: open models | L4 GPU | 0 with `SESSION = "open"`, then 8c, then 7 (the report) |
| 6. Scoring and reading | CPU | 0 with `SESSION = "read"`, then 10, then 11 |

Cells 0b to 6 are done and refuse to run again.

**Rules.** One runner. The scripts print counts and ids only; cell 1 prints question text and no model answer. Every forced-choice command runs twice: the second run must print `nothing to run: 0 calls` (Gate C, D158). Forced choice starts only after Cell 8a passed with this script version (D164). If a cell stops, send its whole output to the Pilot reading chat; do not change a setting yourself.

**Who opens what (D168).** Drive `scored/pilot/` holds the scorer's labels of the pilot answers. The two people who will be the HT-4 checkers do not open Drive `scored/` until both HT-4 sheets are in.

**Outputs hold question text.** Cell 9 removes all outputs from this notebook before it commits it; never commit a notebook by another route.

**Before step 2, once:** the design file `BanglaIndoorVQA_Design.md` (v2.1) is in Drive `BanglaIndoorVQA/docs/`, and the Colab secrets `GITHUB_TOKEN`, `GEMINI_API_KEY` (session A) and `HF_TOKEN` (session B; in session A it lets cell 0 check the Hub commits) are shared with this notebook.

### Cell 0. Set-up and input check (every session)
Set `SESSION` first: `"gemini"` for session A (any runtime), `"open"` for session B (L4 GPU), `"read"` for scoring and reading (CPU). The last line must read `ALL INPUTS PASS`. A FAIL line stops the session. Lines C17 to C24 check the Pilot reading: the 28-row excluded list, the option file and P-FCE.json of ASSET-11 (by sha256), the forced-choice items, the scorer, the frozen lexicon, the outputs to score and, in session B, the environment the pilot stage was fixed to.

In [ ]:
# Cell 0 (every session): set-up, then one PASS or FAIL line per input.
SESSION = "gemini"            # session A: "gemini" (any runtime); session B: "open" (L4 GPU); scoring and reading: "read" (CPU)
PAID_TIER_CONFIRMED = True     # session A: True only after the AI Studio Projects page shows a paid tier for this key's project
PRICES_CHECKED = "2026-10-14"   # the day Google's model and pricing pages were last read (Design 0.2 rule 5); at most 7 days before a billed cell

GITHUB_REPO = "blkeyd/banglaindoorvqa-main"
ROOT = "/content/drive/MyDrive/BanglaIndoorVQA"
REPO = "/content/banglaindoorvqa-main"
DESIGN_MD = ROOT + "/docs/BanglaIndoorVQA_Design.md"                # design v2.1, for the code-point check
ENGLISH = ROOT + "/english/english_questions_final_PILOT.csv"       # version 2; never replaced once the pilot ran

import os, subprocess, sys
from google.colab import drive, userdata

if SESSION not in ("gemini", "open", "read"):
    raise SystemExit('SESSION must be "gemini", "open" or "read".')
drive.mount("/content/drive")
_HIDE = []

def secret(name, required):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if not value and required:
        raise SystemExit(f"STOP: the Colab secret {name} is missing or not shared with this notebook.")
    if value:
        _HIDE.append(value)
    return value or ""

def run(*args, check=True):
    """Run one command, show its output as it comes, stop the cell if it fails."""
    env = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8", PYTHONDONTWRITEBYTECODE="1")
    p = subprocess.Popen([str(x) for x in args], cwd=REPO if os.path.isdir(REPO) else "/content",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        for s in _HIDE:
            line = line.replace(s, "***")
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise SystemExit(f"STOPPED with exit code {p.returncode}. Send the output above to the Pilot chat.")
    return p.returncode

token = secret("GITHUB_TOKEN", True)
if not os.path.isdir(REPO):
    run("git", "clone", "-q", f"https://{token}@github.com/{GITHUB_REPO}.git", REPO)
run("git", "-C", REPO, "pull", "-q", "--ff-only")
run("git", "-C", REPO, "fetch", "-q", "--tags")             # the tag gate-A, for line C22
run("git", "-C", REPO, "log", "-1", "--format=repo at commit %h, %cd", "--date=iso")

os.environ["HF_TOKEN"] = secret("HF_TOKEN", SESSION == "open")
if SESSION == "gemini":
    os.environ["GEMINI_API_KEY"] = secret("GEMINI_API_KEY", True)
elif SESSION == "open":
    run(sys.executable, "-m", "pip", "install", "-q", "flash-linear-attention", "transformers==5.18.0")   # D152, D153

run(sys.executable, "scripts/pilot.py", "check", "--session", {"gemini": "gemini", "open": "open", "read": "any"}[SESSION],
    "--root", ROOT, "--design", DESIGN_MD, "--english-file", ENGLISH)

### (Done 2026-10-14) Cell 0b. Session A, once: the exclusions (D160, D161), no cost
Lists the probe items to exclude under D110: H68_0023_q2 (the gold is a typing slip) and every item of P-CORE-CLOSED, R-TEST, P-NAME and P-CLUST whose Bangla question holds the word for "marked", which points at a box the models never see. Read from the question text and the gold only. First run it as it is and read the list. If it is right, set `WRITE_EXCLUSIONS = True`, run it again, then run cell 9 with `UPLOAD_FROM_COMPUTER = False` and the message `Pilot: excluded items, D160 and D161`. Then run cell 0 again. If the list shows "other forms of the root word", stop and bring them to the Pilot chat before writing.

In [ ]:
# Cell 0b (session A, once): the exclusions of D160 and D161. Question text and gold only; no cost.
DONE = True   # this cell ran on 2026-10-14 (DESIGN_ADDENDUM F). Set False only if the Pilot reading chat asks.
if DONE:
    raise SystemExit("Done on 2026-10-14: not run again (see the order of work at the top).")
WRITE_EXCLUSIONS = False   # first False: read the list; then True: append the rows to probes/excluded_items.csv
write = ["--write"] if WRITE_EXCLUSIONS else []
run(sys.executable, "scripts/pilot.py", "exclusions", *write, "--root", ROOT, "--english-file", ENGLISH)

### (Done 2026-10-14) Cell 1. Session A: the English questions, before any English call (no cost)
Shows the Bangla and English question of the 3 items the first English calls use, and one full prompt per condition. Read each English question against its Bangla one. A wrong English question: stop and bring its qa_id to the Pilot chat.

In [ ]:
# Cell 1 (session A): look at the English questions and the prompts. Question text only, no answer, no cost.
DONE = True   # this cell ran on 2026-10-14 (DESIGN_ADDENDUM F). Set False only if the Pilot reading chat asks.
if DONE:
    raise SystemExit("Done on 2026-10-14: not run again (see the order of work at the top).")
run(sys.executable, "scripts/pilot.py", "preview", "--root", ROOT, "--english-file", ENGLISH)

### (Done 2026-10-14) Cell 2. Session A: dry run of every pilot command (no network, no GPU, no cost)
480 work items per model through the real PILOT.json, English file and images, with made-up answers. Every command must end with `DRY RUN OK`. The dry report at the end shows the report format; its numbers are made up.

In [ ]:
# Cell 2 (session A): dry run of every pilot command for the three models. No cost.
DONE = True   # this cell ran on 2026-10-14 (DESIGN_ADDENDUM F). Set False only if the Pilot reading chat asks.
if DONE:
    raise SystemExit("Done on 2026-10-14: not run again (see the order of work at the top).")
for model in ("gemini", "qwen", "gemma"):
    for i, group in enumerate(("exp-l", "exp-n-open")):
        fresh = ["--dry-fresh"] if i == 0 else []
        run(sys.executable, "scripts/pilot.py", "run", "--model", model, "--group", group,
            "--dry-run", *fresh, "--root", ROOT, "--english-file", ENGLISH)
run(sys.executable, "scripts/pilot.py", "report", "--dry-run", "--root", ROOT, "--english-file", ENGLISH)

### (Done 2026-10-14) Cell 3. Session A: the first billed calls (9 calls, about 0.01 USD)
L-EB, L-EE and L-EE0 on the 3 items of cell 1. First set `PAID_TIER_CONFIRMED = True` in cell 0 after looking at the AI Studio Projects page, and run cell 0 again. The harness checks that `gemini-3.6-flash` is served before the first call.

In [ ]:
# Cell 3 (session A): Gemini, the English conditions on 3 items first. Billed.
DONE = True   # this cell ran on 2026-10-14 (DESIGN_ADDENDUM F). Set False only if the Pilot reading chat asks.
if DONE:
    raise SystemExit("Done on 2026-10-14: not run again (see the order of work at the top).")
if not PAID_TIER_CONFIRMED:
    raise SystemExit("STOP: look at the AI Studio Projects page, set PAID_TIER_CONFIRMED = True in cell 0, run cell 0 again.")
run(sys.executable, "scripts/pilot.py", "run", "--model", "gemini", "--group", "english3",
    "--paid-tier-confirmed", "--prices-checked", PRICES_CHECKED, "--root", ROOT, "--english-file", ENGLISH)

### (Done 2026-10-14) Cell 4. Session A: Gemini, the rest of the 9 conditions (471 calls, about 0.35 USD)
EXP-L on the 60 closed items (351 calls, 9 are in the cache from cell 3), then EXP-N open naming on the 40 naming items (120 calls). Each command runs twice. About 15 to 25 minutes.

In [ ]:
# Cell 4 (session A): Gemini, EXP-L and EXP-N open naming. Billed.
DONE = True   # this cell ran on 2026-10-14 (DESIGN_ADDENDUM F). Set False only if the Pilot reading chat asks.
if DONE:
    raise SystemExit("Done on 2026-10-14: not run again (see the order of work at the top).")
if not PAID_TIER_CONFIRMED:
    raise SystemExit("STOP: PAID_TIER_CONFIRMED is False (cell 0).")
for group in ("exp-l", "exp-n-open"):
    run(sys.executable, "scripts/pilot.py", "run", "--model", "gemini", "--group", group,
        "--paid-tier-confirmed", "--prices-checked", PRICES_CHECKED, "--root", ROOT, "--english-file", ENGLISH)

### (Done 2026-10-14) Cell 5. Session A: the D5 repeat (D159, approved)
Run it right after cell 4, in the same session, so the two runs are minutes apart. 100 calls, about 0.09 USD, in stage smoke (run name `pilot-repeat`). Then two comparisons, counts only: against the pilot answers (same mode, minutes apart) and against the Batch test of 2026-10-05 (no cost).

In [ ]:
# Cell 5 (session A): the D5 repeat, approved as D159. Billed, about 0.09 USD.
DONE = True   # this cell ran on 2026-10-14 (DESIGN_ADDENDUM F). Set False only if the Pilot reading chat asks.
if DONE:
    raise SystemExit("Done on 2026-10-14: not run again (see the order of work at the top).")
D5_APPROVED = True
if D5_APPROVED and PAID_TIER_CONFIRMED:
    for group in ("d5-closed", "d5-naming"):
        run(sys.executable, "scripts/pilot.py", "run", "--model", "gemini", "--group", group, "--d5-approved",
            "--paid-tier-confirmed", "--prices-checked", PRICES_CHECKED, "--root", ROOT, "--english-file", ENGLISH)
    for other in ("smoke/pilot-repeat:gemini", "smoke/batchtest:gemini"):
        run(sys.executable, "scripts/pilot.py", "compare", "--a", "pilot:gemini", "--b", other,
            "--conditions", "L-BB,N-BB", "--d5-approved", "--root", ROOT)
else:
    print("D5 repeat is off (D5_APPROVED = False).")

### (Done 2026-10-14) Cell 6. Session B (L4): Qwen3.5-4B, then Gemma 4 E4B
Run cell 0 first with `SESSION = "open"` on an L4 runtime. For each model: the English conditions on 3 items, then EXP-L, then EXP-N open naming; each command twice. Keep all of one model's commands in this one session: the first pilot run fixes the environment of the stage (D147). About 30 to 60 minutes for both models.

In [ ]:
# Cell 6 (session B, L4): the two open models. No API cost.
DONE = True   # this cell ran on 2026-10-14 (DESIGN_ADDENDUM F). Set False only if the Pilot reading chat asks.
if DONE:
    raise SystemExit("Done on 2026-10-14: not run again (see the order of work at the top).")
if SESSION != "open":
    raise SystemExit('STOP: this cell needs SESSION = "open" and an L4 runtime (run cell 0 again).')
for model in ("qwen", "gemma"):
    for group in ("english3", "exp-l", "exp-n-open"):
        run(sys.executable, "scripts/pilot.py", "run", "--model", model, "--group", group,
            "--root", ROOT, "--english-file", ENGLISH)

### Cell 7. The pilot report (either session, after cells 8b and 8c)
Counts only. Paste the whole output into the Pilot reading chat. After the forced-choice runs it shows the four N-FC rows (20 of 20 each), the Gemini tokens of forced choice, and the Gate C first-half lines of `exp-n-fcb` and `exp-n-fce`. It is also saved in Drive `outputs/pilot/pilot_report.txt` and `.json`.

In [ ]:
# Cell 7: the pilot report. Counts, seconds per item, tokens per call, projections against Design 12.
run(sys.executable, "scripts/pilot.py", "report", "--root", ROOT, "--english-file", ENGLISH)

### Cell 8a. Session A: forced choice as a dry run on the real files, then fc-verify (no network, no GPU, no cost)
The four forced-choice conditions for the three models with made-up answers, through the real PILOT.json, option file, P-FCE.json, English file and images. Every command must end with `DRY RUN OK`. Then `fc-verify` checks that every forced-choice item is in each dry cache once, and that each prompt, rebuilt by the harness from the option file, has the cached hash and shows the scorer's four options in the order of the file under the right labels. It must end `FC-VERIFY: PASS`; cells 8b and 8c refuse to run before that (D164). A `SKIP` line means a harness helper was called in another way than pilot.py expects: send the output and `src/harness.py` to the chat.

In [ ]:
# Cell 8a (session A): forced choice, dry run on the real files for the three models, then fc-verify. No cost.
OPTIONS = ROOT + "/probes/fc_options.jsonl"   # the final ASSET-11 file (sha256 5288ba63...); pilot.py refuses any other (D164)
PFCE = ROOT + "/probes/P-FCE.json"            # 454 items (sha256 807e987b...)
for model in ("gemini", "qwen", "gemma"):
    run(sys.executable, "scripts/pilot.py", "run", "--model", model, "--group", "exp-n-fcb", "--options-file", OPTIONS,
        "--dry-run", "--dry-fresh", "--root", ROOT, "--english-file", ENGLISH)
    run(sys.executable, "scripts/pilot.py", "run", "--model", model, "--group", "exp-n-fce", "--options-file", OPTIONS,
        "--pfce", PFCE, "--dry-run", "--root", ROOT, "--english-file", ENGLISH)
run(sys.executable, "scripts/pilot.py", "fc-verify", "--dry-run", "--root", ROOT, "--english-file", ENGLISH)


### Cell 8b. Session A: Gemini, the four forced-choice conditions (80 calls, about 0.04 USD)
N-FCB and N-FCB0 on the 20 naming items with `in_pclust = true`, then N-FCE and N-FCE0 on the 20 naming items in P-FCE (stage pilot, standard mode). Each command runs twice; the second run must make 0 calls. The harness checks that `gemini-3.6-flash` version 3.6-flash-07-2026 is served before the first call and keeps the stage under its 2.00 USD cap (0.3152 USD spent before this cell). `PRICES_CHECKED` in cell 0 must be at most 7 days old.

In [ ]:
# Cell 8b (session A): Gemini, the four forced-choice conditions. Billed, about 0.04 USD. Each command runs twice (D158).
if SESSION != "gemini":
    raise SystemExit('STOP: this cell needs SESSION = "gemini" (run cell 0 again).')
if not PAID_TIER_CONFIRMED:
    raise SystemExit("STOP: PAID_TIER_CONFIRMED is False (cell 0).")
OPTIONS = ROOT + "/probes/fc_options.jsonl"
PFCE = ROOT + "/probes/P-FCE.json"
billed = ["--paid-tier-confirmed", "--prices-checked", PRICES_CHECKED]
run(sys.executable, "scripts/pilot.py", "run", "--model", "gemini", "--group", "exp-n-fcb", "--options-file", OPTIONS,
    *billed, "--root", ROOT, "--english-file", ENGLISH)
run(sys.executable, "scripts/pilot.py", "run", "--model", "gemini", "--group", "exp-n-fce", "--options-file", OPTIONS,
    "--pfce", PFCE, *billed, "--root", ROOT, "--english-file", ENGLISH)


### Cell 8c. Session B (L4): Qwen3.5-4B, then Gemma 4 E4B, the four forced-choice conditions
Run cell 0 first with `SESSION = "open"` on an L4 runtime; line C24 must not FAIL (the session must match the environment the pilot stage was fixed to on 2026-10-14, D147). For each model: `exp-n-fcb`, then `exp-n-fce`; each command twice. About 10 minutes for both models.

In [ ]:
# Cell 8c (session B, L4): the two open models, the four forced-choice conditions. No API cost.
if SESSION != "open":
    raise SystemExit('STOP: this cell needs SESSION = "open" and an L4 runtime (run cell 0 again).')
OPTIONS = ROOT + "/probes/fc_options.jsonl"
PFCE = ROOT + "/probes/P-FCE.json"
for model in ("qwen", "gemma"):
    run(sys.executable, "scripts/pilot.py", "run", "--model", model, "--group", "exp-n-fcb", "--options-file", OPTIONS,
        "--root", ROOT, "--english-file", ENGLISH)
    run(sys.executable, "scripts/pilot.py", "run", "--model", model, "--group", "exp-n-fce", "--options-file", OPTIONS,
        "--pfce", PFCE, "--root", ROOT, "--english-file", ENGLISH)


### Cell 10. Scoring (session "read", CPU, no cost)
Scores every pilot answer with scorer 0.2.1-stage2 into Drive `scored/pilot/`: one `asset9_scorer.py score` command per cache (the three pilot caches, the D5 repeat, the Batch test of 2026-10-05), never two caches in one command (D165). It refuses to start while a forced-choice answer is missing. Expected: 560 records per pilot model (19 excluded, 541 scored), 100 in the D5 repeat and 100 in the Batch test (3 excluded, 97 scored). Running it again keeps files scored from the same inputs.

In [ ]:
# Cell 10 (session "read", CPU): score every pilot answer with scorer 0.2.1-stage2, one command per cache (D165).
OVERWRITE = False   # True only when the Pilot reading chat asks for it
run(sys.executable, "scripts/pilot.py", "score", "--root", ROOT, "--english-file", ENGLISH,
    *(["--overwrite"] if OVERWRITE else []))


### Cell 11. The Pilot reading (session "read", no cost)
Counts and ids only; paste the whole output into the Pilot reading chat. Gate C (both halves, D158, D163, D190), Gate D (D163), the D5 agreement on 97 pairs (D167), refusal rates without the image, correct answers that hold a refusal phrase (D194), forced choice, the prompt check on the real caches, and the 10-item check. Then upload Drive `scored/pilot/pilot_reading_sample.csv` to the chat for the 10-item check (it holds answer text: only the runner opens it, D168).

In [ ]:
# Cell 11 (session "read"): the Pilot reading. Counts and ids only; text goes to Drive scored/pilot/*.csv.
run(sys.executable, "scripts/pilot.py", "read", "--root", ROOT, "--english-file", ENGLISH)


### Cell 9. Commit and push to the repo (any session)
Step 1 of the Pilot reading: `UPLOAD_FROM_COMPUTER = True`, choose `pilot.py` and `pilot.ipynb` of the Pilot reading chat (Colab may add " (1)" to a name, which is fine). The reading itself writes only to Drive, so nothing else is committed. Only these four repo paths are ever committed: `scripts/pilot.py`, `notebooks/pilot.ipynb`, `configs/models.yaml`, `probes/excluded_items.csv`. Notebook outputs are removed first. Fill in `GIT_NAME` and `GIT_EMAIL` once.

In [ ]:
# Cell 9 (any session): commit and push. Code, configs and the excluded-items list only; never data,
# question text or model output.
UPLOAD_FROM_COMPUTER = False   # True: choose files on your computer. False: commit what changed in the clone.
COMMIT_MESSAGE = "Pilot reading: forced-choice guards, scoring and reading (pilot-2026-10-16a)"
GIT_NAME = "Arafat"                 # your GitHub user name
GIT_EMAIL = "ar.hossain1303@gmail.com"                # the e-mail address of your GitHub account

import json, os, re, subprocess
from google.colab import files, userdata

GITHUB_REPO = "blkeyd/banglaindoorvqa-main"
REPO = "/content/banglaindoorvqa-main"
TARGETS = {"pilot.py": "scripts/pilot.py", "pilot.ipynb": "notebooks/pilot.ipynb",
           "models.yaml": "configs/models.yaml", "excluded_items.csv": "probes/excluded_items.csv"}
if not GIT_NAME.strip() or not GIT_EMAIL.strip():
    raise SystemExit("STOP: fill in GIT_NAME and GIT_EMAIL first. Nothing was committed.")
_tok = userdata.get("GITHUB_TOKEN")

def git(*args, check=True, quiet=False):
    p = subprocess.run(["git", "-C", REPO] + [str(a) for a in args], capture_output=True, text=True)
    out = (p.stdout + p.stderr).replace(_tok, "***")
    if out.strip() and not quiet:
        print(out.rstrip())
    if check and p.returncode != 0:
        raise SystemExit(f"STOPPED: git {args[0]} failed. Nothing more was done. Send the output above to the Pilot chat.")
    return p.stdout

if not os.path.isdir(REPO):
    c = subprocess.run(["git", "clone", "-q", f"https://{_tok}@github.com/{GITHUB_REPO}.git", REPO],
                       capture_output=True, text=True)
    if c.returncode != 0:
        raise SystemExit("STOPPED: git clone failed: " + c.stderr.replace(_tok, "***"))
git("pull", "-q", "--ff-only")

if UPLOAD_FROM_COMPUTER:
    for name, data in files.upload().items():
        base = re.sub(r" \(\d+\)(?=\.[^.]+$)", "", os.path.basename(name))
        if base not in TARGETS:
            raise SystemExit(f"STOP: {name} is not one of {sorted(TARGETS)}. Nothing was committed.")
        target = os.path.join(REPO, TARGETS[base])
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with open(target, "wb") as f:
            f.write(data)
        print(f"placed {name} as {TARGETS[base]} ({len(data):,} bytes)")

nb_path = os.path.join(REPO, TARGETS["pilot.ipynb"])
if os.path.exists(nb_path):                      # outputs hold question text: never commit them
    with open(nb_path, encoding="utf-8") as f:
        nb = json.load(f)
    code_cells = [c for c in nb.get("cells", []) if c.get("cell_type") == "code"]
    if any(c.get("outputs") or c.get("execution_count") is not None for c in code_cells):
        for c in code_cells:
            c["outputs"], c["execution_count"] = [], None
        (nb.get("metadata") or {}).pop("widgets", None)
        with open(nb_path, "w", encoding="utf-8") as f:
            json.dump(nb, f, ensure_ascii=False, indent=1)
        print("removed the outputs from notebooks/pilot.ipynb")

changed = [line[3:].strip() for line in git("status", "--porcelain", "--untracked-files=all").splitlines()]
mine = [p for p in changed if p in TARGETS.values()]
other = [p for p in changed if p not in TARGETS.values()]
if other:
    print("NOT committed (outside the four paths of this chat): " + ", ".join(other)
          + ". If one of them is a tracked file, run git checkout on it before cell 0 (C01 needs a clean tree).")
if not mine:
    print("Nothing to commit: the repo already holds these files.")
else:
    for p in mine:
        if os.path.getsize(os.path.join(REPO, p)) > 2_000_000:
            raise SystemExit(f"STOP: {p} is over 2 MB; that is not a code file. Nothing was committed.")
    if "probes/excluded_items.csv" in mine:      # rows may only be added (DESIGN_ADDENDUM C4)
        before = git("show", "HEAD:probes/excluded_items.csv", check=False, quiet=True)
        with open(os.path.join(REPO, "probes/excluded_items.csv"), encoding="utf-8") as f:
            now = f.read()
        ids = [line.split(",")[0] for line in now.splitlines()[1:] if line.strip()]
        if not now.startswith(before) or len(ids) != len(set(ids)):
            raise SystemExit("STOP: probes/excluded_items.csv must only grow by new rows, each qa_id once. "
                             "Nothing was committed. Send this output to the Pilot chat.")
    git("add", "--", *mine)
    git("diff", "--cached", "--stat")
    if "probes/excluded_items.csv" in mine:
        git("diff", "--cached", "--unified=0", "--", "probes/excluded_items.csv")   # qa_ids and codes only
    git("-c", f"user.name={GIT_NAME}", "-c", f"user.email={GIT_EMAIL}", "commit", "-q", "-m", COMMIT_MESSAGE)
    git("push", "-q")
    git("log", "-1", "--format=pushed commit %h: %s")